# Grid Contingency Analysis
Which single failures hurt the grid? How does a failure spread? 

Import necessary packages.

In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp
import pandapower.networks as pn
import pandapower.plotting
import pandapower.converter
import pandapower.estimation

In [23]:
# load benchmark IEEE 14-bus network
net = pn.power_system_test_cases.case14()

# run standard AC power flow
pp.runpp(net)

# get all transmission lines, put in array
lines = net.line.index.tolist()
print(f"We will test {len(lines)} transmission lines.")

We will test 15 transmission lines.


Establish the N-0 case for the IEEE 14-bus system. This serves as a baseline for comparison for N-K contingencies.

In [24]:
# N-0 case (for comparison)
v_min, v_max, load_max = 0.95, 1.10, 100 # establish thresholds, v_max increased to account for higher max bus voltage
net.line['in_service'] = True # set line to active

# run power flow (pre-correction)
pp.runpp(net)
print(net.line['max_i_ka'])
print(net.res_line['loading_percent'].describe())

# override unusually large current values with 150% of base current
net.line['max_i_ka'] = net.res_line['i_ka'] * 1.5

# establish new baseline power flow
pp.runpp(net)
print("Corrected N-0 max line loading")
print(f"{net.res_line['loading_percent'].max():.2f}%")

# extract critical values
base_min_v = net.res_bus['vm_pu'].min()
base_max_v = net.res_bus['vm_pu'].max()
base_ll_max = net.res_line['loading_percent'].max()
base_islanded = int(net.res_bus['vm_pu'].isna().sum())

if base_ll_max > load_max or base_min_v < v_min or base_max_v > v_max or base_islanded > 0:
    status = "Violation"
else:
    status = "Pass"

base_case = pd.DataFrame([{
    "Tripped_Line": "N/A",
    "Status": status,
    "Max_Line_Loading_%": base_ll_max,
    "Min_Bus_V_pu": base_min_v,
    "Max_Bus_V_pu": base_max_v,
    "Islanded_Buses": base_islanded,    
}])

display(base_case)

0        42.339020
1        42.339020
2        42.339020
3        42.339020
4        42.339020
5        42.339020
6        42.339020
7     27479.652235
8     27479.652235
9     27479.652235
10    27479.652235
11    27479.652235
12    27479.652235
13    27479.652235
14    27479.652235
Name: max_i_ka, dtype: float64
count    15.000000
mean      0.357327
std       0.409831
min       0.017055
25%       0.070694
50%       0.180865
75%       0.584899
max       1.507568
Name: loading_percent, dtype: float64
Corrected N-0 max line loading
66.67%


,Tripped_Line,Status,Max_Line_Loading_%,Min_Bus_V_pu,Max_Bus_V_pu,Islanded_Buses
0,N/A,Pass,66.666667,1.01,1.09,0


Simulate a fault at each line. Find any thermal/voltage violations. Tabulate the results.

In [25]:
# N-1 case
n1_rows = []

for i in lines:
    # simulate a fault/outage at a given transmission line
    net.line.loc[i, 'in_service'] = False

# exception handling to manage the result of Newton-Ralphson method
    try:
        pp.runpp(net) # recalculate power flow

        # net.res_bus for result params
        # check: undervoltage and overvoltage violations
        min_bus_voltage = net.res_bus['vm_pu'].min() # 'vm_pu': voltage magnitude at connection bus
        max_bus_voltage = net.res_bus['vm_pu'].max()

        # check: large and unusual angle deviations
        bus_voltage_angle = net.res_bus['va_degree'] # 'va_degree': voltage angle [deg]

        # active and reactive power
        power_a = net.res_bus['p_mw']
        power_r = net.res_bus['q_mvar']

        # check: thermal overload/voltage sag
        ll_max = net.res_line['loading_percent'].max()

        # check
        n_islanded = int(net.res_bus['vm_pu'].isna().sum()) 

        # performance index (PI)
        n = 1.0 # sensitivity, n=1 is a common starting value
        perf_index_v = ((abs(net.res_bus['vm_pu']-1.0)/0.10)**(2*n)).sum() # delta V represents bus voltage deviation from nominal target voltage (1.0 pu)
        perf_index_l = ((net.res_line['loading_percent']/100)**(2*n)).sum() # divide by 100 to get ratio
        perf_index_total = perf_index_v + perf_index_l

        # extend upper voltage limit (default 1.09)
        if ll_max > load_max or min_bus_voltage < v_min or max_bus_voltage > v_max or n_islanded > 0:
            status = "Violation"
        else:
            status = "Pass"


    except pp.LoadflowNotConverged:
        min_bus_voltage = max_bus_voltage = ll_max = float("nan")
        n_islanded = None
        status = "Non-convergence"

    finally:
        net.line.loc[i, 'in_service'] = True # restore the line

    n1_rows.append({
        "Tripped_Line": i,
        "Status": status,
        "Max_Line_Loading_%": ll_max,
        "Min_Bus_V_pu": min_bus_voltage,
        "Max_Bus_V_pu": max_bus_voltage,
        "Islanded_Buses": n_islanded,
        "PI_V": perf_index_v,
        "PI_L": perf_index_l,
        "PI_Total": perf_index_total,
        # "Active_Power_MW": power_a,
        # "Reactive_Power_MVar": power_r, 

    })

# append results to table
results = pd.DataFrame(n1_rows)
display(results)

,Tripped_Line,Status,Max_Line_Loading_%,Min_Bus_V_pu,Max_Bus_V_pu,Islanded_Buses,PI_V,PI_L,PI_Total
0,0,Violation,232.333140,0.993484,1.09,0,3.473767,16.756938,20.230704
1,1,Violation,124.744829,1.006442,1.09,0,3.706091,7.884805,11.590896
2,2,Violation,293.137559,1.010000,1.09,0,3.704719,17.717623,21.422342
3,3,Violation,107.065276,1.007096,1.09,0,3.620727,8.335375,11.956102
4,4,Pass,88.523691,1.010000,1.09,0,3.759025,6.198989,9.958014
5,5,Pass,89.372204,1.010000,1.09,0,3.983500,5.903667,9.887167
6,6,Violation,218.873443,1.010000,1.09,0,3.577566,14.558385,18.135952
7,7,Violation,147.875387,1.010000,1.09,0,3.386953,8.766226,12.153179
8,8,Violation,242.407582,1.010000,1.09,0,3.548101,12.131787,15.679888
9,9,Violation,520.767681,0.997979,1.09,0,3.084093,37.108317,40.192410


Sort total PI from greatest to least to identify the most vulnerable lines.

In [26]:
display(results.sort_values(by="PI_Total",ascending=False))

,Tripped_Line,Status,Max_Line_Loading_%,Min_Bus_V_pu,Max_Bus_V_pu,Islanded_Buses,PI_V,PI_L,PI_Total
9,9,Violation,520.767681,0.997979,1.09,0,3.084093,37.108317,40.192410
2,2,Violation,293.137559,1.010000,1.09,0,3.704719,17.717623,21.422342
0,0,Violation,232.333140,0.993484,1.09,0,3.473767,16.756938,20.230704
6,6,Violation,218.873443,1.010000,1.09,0,3.577566,14.558385,18.135952
11,11,Violation,186.864417,0.996870,1.09,0,3.904403,11.866519,15.770922
8,8,Violation,242.407582,1.010000,1.09,0,3.548101,12.131787,15.679888
10,10,Violation,177.773334,1.010000,1.09,0,3.771260,9.660543,13.431803
7,7,Violation,147.875387,1.010000,1.09,0,3.386953,8.766226,12.153179
3,3,Violation,107.065276,1.007096,1.09,0,3.620727,8.335375,11.956102
1,1,Violation,124.744829,1.006442,1.09,0,3.706091,7.884805,11.590896


Devise a strategy to mitigate the effects of vunlnerable line outages.